# Tugas Individu Informed Search

**Nama:** Earlang Rangga Purwanto  
**NRP:** 5025251054  

## 1. Tujuan dan Pembagian Algoritma

Buat program untuk mencari rute robot pengantar dari depot **S** menuju gedung **G**.
Setiap jalan membutuhkan biaya energi yang berbeda.

| Digit terakhir NRP | Algoritma |
|---|---|
| Ganjil: 1, 3, 5, 7, 9 | Greedy Best-First Search |
| Genap: 0, 2, 4, 6, 8 | A* Search |

Kerjakan **satu algoritma sesuai NRP**. Anda akan mengimplementasikan search,
menampilkan prosesnya, lalu menjelaskan hasil eksperimen.

**Bentuk tugas:** graph dictionary dengan satu tujuan, mengikuti konsep search CS188
yang disederhanakan. Tidak perlu membuat class, engine Pacman, multi-goal problem,
atau menjalankan autograder Berkeley.

## 2. Problem dan Data

Graph memiliki 14 state dengan beberapa alternatif rute dan cycle.

- Graph berarah: hanya jalan yang ditulis yang boleh digunakan.
- Pasangan `("A", 3)` berarti jalan ke A berbiaya energi 3.
- Heuristic memperkirakan energi minimum yang masih diperlukan menuju G.
- Heuristic awal nonnegatif, consistent, dan h(G)=0.
- Data graph dan heuristic awal dipakai untuk **baseline**; jangan diubah.

In [1]:
import math
import heapq
from itertools import count

graph = {
    "S": [("A", 3), ("B", 5), ("C", 4)],
    "A": [("D", 4), ("E", 7), ("B", 1)],
    "B": [("E", 2), ("F", 6)],
    "C": [("B", 2), ("F", 3), ("H", 5)],
    "D": [("I", 4), ("E", 2)],
    "E": [("D", 3), ("I", 5), ("J", 6)],
    "F": [("E", 2), ("J", 3), ("K", 7)],
    "H": [("F", 2), ("K", 3)],
    "I": [("L", 3), ("J", 2)],
    "J": [("I", 2), ("L", 5), ("M", 4)],
    "K": [("J", 2), ("M", 3)],
    "L": [("G", 6), ("M", 2)],
    "M": [("G", 3)],
    "G": [],
}

heuristic = {
    "S": 9, "A": 8, "B": 7, "C": 7,
    "D": 7, "E": 6, "F": 5, "H": 5,
    "I": 5, "J": 4, "K": 4, "L": 3, "M": 2, "G": 0,
}
start = "S"
goal = "G"
print("Start State:", start)
print("Goal State:", goal)
print("Jumlah state:", len(graph))
print("Jumlah edge:", sum(len(successors) for successors in graph.values()))
for state, successors in graph.items():
    print(f"{state}: {successors}, h={heuristic[state]}")

Start State: S
Goal State: G
Jumlah state: 14
Jumlah edge: 31
S: [('A', 3), ('B', 5), ('C', 4)], h=9
A: [('D', 4), ('E', 7), ('B', 1)], h=8
B: [('E', 2), ('F', 6)], h=7
C: [('B', 2), ('F', 3), ('H', 5)], h=7
D: [('I', 4), ('E', 2)], h=7
E: [('D', 3), ('I', 5), ('J', 6)], h=6
F: [('E', 2), ('J', 3), ('K', 7)], h=5
H: [('F', 2), ('K', 3)], h=5
I: [('L', 3), ('J', 2)], h=5
J: [('I', 2), ('L', 5), ('M', 4)], h=4
K: [('J', 2), ('M', 3)], h=4
L: [('G', 6), ('M', 2)], h=3
M: [('G', 3)], h=2
G: [], h=0


### Peta pekerjaan: mulai dari sini

| Urutan | Yang Anda lakukan | Tanda selesai |
|---|---|---|
| 1 | Baca bagian 2–6 materi dan aturan di bawah | Dapat membedakan g, h, f, dan struktur frontier |
| 2 | Jalankan data dan helper yang disediakan | Tidak ada error; belum ada solusi |
| 3 | Isi rekonstruksi path (R1–R2) | Uji parent mini lulus |
| 4 | Isi G1–G8 atau A1–A9 sesuai NRP | Return sesuai kontrak dan trace terisi |
| 5 | Jalankan uji mini | Semua kasus untuk algoritma Anda lulus |
| 6 | Jalankan baseline S menuju G | Path legal, cost sesuai, trace dapat dijelaskan |
| 7 | Isi analisis dan perubahan heuristic | Dua run dapat dibandingkan |
| 8 | Restart Kernel dan Run All, lalu simpan | Tidak ada error atau pesan TODO tersisa |


## 3. Aturan Implementasi

### a. Aturan bersama

1. Gunakan `heapq` untuk frontier.
2. Jika priority sama, pilih kandidat yang masuk lebih dahulu (FIFO).
3. Proses successor sesuai urutan pada list graph.
4. Gunakan dictionary `parent` untuk menyusun path.
5. Hitung cost dari jumlah bobot edge, bukan dari heuristic.

**Ekspansi** berarti memproses successor. Goal dan entri usang tidak dihitung.
Jika A* mengekspansi ulang state, peristiwa tersebut dihitung lagi.

### b. Aturan sesuai algoritma

| Greedy | A* |
|---|---|
| Priority = h | Priority = g+h |
| Simpan path pertama ke setiap state | Simpan g terbaik yang telah ditemukan |
| Jangan masukkan ulang state yang sudah ditemukan | Update g dan parent jika jalur baru lebih murah |
| Catat state yang sudah diekspansi | Lewati entri usang; dukung pembukaan ulang |

Periksa goal saat diambil dari frontier. Pada A*, periksa entri usang terlebih dahulu.
Kedua algoritma harus berhenti dengan hasil gagal jika frontier habis tanpa mencapai goal.

### c. Format hasil fungsi

Kembalikan dictionary berikut:

```python
{
    "path": [...],
    "cost": ...,
    "expanded": ...,
    "expansion_order": [...],
    "trace": [...],
}
```

| Field | Isi |
|---|---|
| `path` | Urutan state dari start sampai goal |
| `cost` | Total bobot edge pada path |
| `expanded` | Jumlah peristiwa ekspansi |
| `expansion_order` | Urutan state yang diekspansi |
| `trace` | Catatan pemilihan state dan frontier |

Jika tidak ada solusi: path `[]`, cost `float("inf")`.
Jika start sama dengan goal: path `[start]`, cost 0, expanded 0.

### d. Format trace

Setiap baris `trace` adalah dictionary dengan field berikut:

| Field | Isi |
|---|---|
| `iteration` | Nomor mulai dari 1 |
| `selected` | State yang dipilih |
| `h` | Heuristic state tersebut |
| `g`, `f` | Tambahan wajib untuk A*; f=g+h |
| `action` | `"expand"` atau `"goal"` |
| `frontier` | String snapshot sesudah successor diproses / goal di-pop |

Tampilkan frontier dalam urutan priority dan FIFO. Format kandidat:
`A(h=...)` untuk Greedy; `A(g=...,h=...,f=...)` untuk A*.
Pada A*, tampilkan kandidat aktif; entri usang boleh dicatat terpisah.

Snapshot harus disimpan sebagai string agar catatan lama tidak berubah saat heap berubah.

**Tambahan A*:** tiap row memuat `updates`, list dictionary pembaruan dengan field
`state`, `previous_g`, `new_g`, `previous_parent`, `new_parent`. Untuk penemuan pertama,
`previous_g=inf` dan `previous_parent=None`; jika tidak ada pembaruan, pakai `[]`.
Field tambahan tingkat hasil `skipped` diperbolehkan, tetapi tidak wajib dinilai.

### e. Contoh bentuk trace (graph mini, bukan jawaban tugas)

```python
{
    "iteration": 1,
    "selected": "R",
    "h": 4,
    "action": "expand",
    "frontier": "T(h=3)"
}
```

Untuk A*, tambahkan `g`, `f`, dan `updates`, serta gunakan format frontier g/h/f.
Jangan menyalin nilai R/T ke graph tugas; contoh hanya memperjelas jenis data.

**Waktu mencatat:** pop valid → buat row → tes goal / proses successor → simpan
snapshot string → append row. Stale pop tidak mendapat nomor iterasi baru.

## 4. Implementasi dan Pengujian

### a. Helper tampilan — disediakan

Cell berikut boleh dipakai langsung. `greedy_frontier_text(frontier)` dan
`astar_frontier_text(frontier, g_cost, heuristic)` hanya memformat snapshot.
`print_trace` dan `show_updates` menampilkan catatan yang dibuat oleh algoritma Anda.
Fungsi-fungsi ini tidak menjalankan search.

In [2]:
def show_table(headers, rows):
    from IPython.display import Markdown, display

    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for row in rows:
        lines.append("| " + " | ".join(str(value) for value in row) + " |")
    display(Markdown("\n".join(lines)))


def print_trace(trace, algorithm):
    headers = ["Iterasi", "Dipilih", "h", "Aksi", "Frontier sesudah"]
    if algorithm == "A*":
        headers = ["Iterasi", "Dipilih", "g", "h", "f", "Aksi", "Frontier sesudah"]
    rows = []
    for row in trace:
        values = [row["iteration"], row["selected"]]
        if algorithm == "A*":
            values.extend([row["g"], row["h"], row["f"]])
        else:
            values.append(row["h"])
        values.extend([row["action"], row["frontier"]])
        rows.append(values)
    show_table(headers, rows)

def greedy_frontier_text(frontier):
    entries = []
    for priority, order, state in sorted(frontier):
        entries.append(f"{state}(h={priority})")
    return "; ".join(entries) or "(kosong)"

def astar_frontier_text(frontier, g_cost, heuristic):
    entries = []
    for priority, order, current_g, state in sorted(frontier):
        if current_g != g_cost[state]:
            continue  # Snapshot hanya menampilkan kandidat aktif, bukan entri usang.
        entries.append(f"{state}(g={current_g},h={heuristic[state]},f={priority})")
    return "; ".join(entries) or "(kosong)"

def show_updates(result):
    rows = []
    for row in result["trace"]:
        for update in row.get("updates", []):
            if update["previous_g"] != float("inf"):
                rows.append([row["iteration"], update["state"],
                             update["previous_g"], update["new_g"],
                             update["previous_parent"], update["new_parent"]])
    show_table(["Iterasi", "State", "g lama", "g baru", "Parent lama", "Parent baru"], rows)

### b. Lengkapi rekonstruksi dan satu algoritma

Ikuti urutan TODO. `problem` berisi adjacency list, sehingga successor dibaca dengan
`for next_state, step_cost in problem[current_state]:`.

| Kelompok | Isi entri heap | Catatan biaya |
|---|---|---|
| Greedy | `(h, nomor_urut, state)` | `path_cost` untuk laporan; jangan memperbarui path pertama |
| A* | `(f, nomor_urut, g, state)` | `g_cost` adalah biaya terbaik yang diketahui |

Gunakan `count()` dan `next(counter)` untuk nomor urut. Jangan menggunakan nama state
sebagai penentu tie. Bedakan list `expanded` dari integer hasil `len(expanded)`.

In [3]:
def reconstruct_path(parent, goal):
    path = []
    current_state = goal

    while current_state is not None:
        path.append(current_state)
        current_state = parent[current_state]

    path.reverse()
    return path


def greedy_search(problem, start, goal, heuristic):
    # TODO G1: Buat frontier, counter FIFO, visited, parent, path_cost, expanded, trace.
    # TODO G2: Set parent[start]=None dan cost awal 0; push (h(start), urutan, start).
    # TODO G3: Selama frontier berisi, pop entri dan buat row trace baru.
    # TODO G4: Jika goal, catat row goal dan return dictionary hasil.
    # TODO G5: Tandai visited dan catat ekspansi; proses problem[current_state].
    # TODO G6: Untuk state baru saja, simpan parent/cost pertama lalu push memakai h.
    # TODO G7: Simpan snapshot sesudah ekspansi dengan greedy_frontier_text.
    # TODO G8: Jika frontier habis, return path=[], cost=inf, beserta statistik.
    pass

import heapq
from itertools import count

def a_star_search(problem, start, goal, heuristic):
    frontier = []
    counter = count()
    g_cost = {}
    parent = {}
    expanded = []
    trace = []

    g_cost[start] = 0
    parent[start] = None

    heapq.heappush(
        frontier,
        (heuristic[start], next(counter), 0, start)
    )

    while frontier:
        f, order, current_g, current_state = heapq.heappop(frontier)

        # Lewati entri lama/stale
        if current_g != g_cost[current_state]:
            continue

        iteration = len(trace) + 1
        h = heuristic[current_state]

        row = {
            "iteration": iteration,
            "selected": current_state,
            "g": current_g,
            "h": h,
            "f": f,
            "action": "expand",
            "frontier": "",
            "updates": [],
        }

        # Goal diperiksa setelah stale entry dilewati
        if current_state == goal:
            row["action"] = "goal"
            row["frontier"] = astar_frontier_text(
                frontier, g_cost, heuristic
            )
            trace.append(row)

            return {
                "path": reconstruct_path(parent, goal),
                "cost": current_g,
                "expanded": len(expanded),
                "expansion_order": expanded,
                "trace": trace,
            }

        # State valid yang dipilih dihitung sebagai ekspansi
        expanded.append(current_state)

        # Proses semua successor sesuai urutan graph
        for next_state, step_cost in problem[current_state]:
            new_g = current_g + step_cost
            previous_g = g_cost.get(next_state, float("inf"))
            previous_parent = parent.get(next_state)

            if new_g < previous_g:
                g_cost[next_state] = new_g
                parent[next_state] = current_state

                new_h = heuristic[next_state]
                new_f = new_g + new_h

                heapq.heappush(
                    frontier,
                    (new_f, next(counter), new_g, next_state)
                )

                row["updates"].append({
                    "state": next_state,
                    "previous_g": previous_g,
                    "new_g": new_g,
                    "previous_parent": previous_parent,
                    "new_parent": current_state,
                })

        # Simpan snapshot frontier setelah ekspansi
        row["frontier"] = astar_frontier_text(
            frontier, g_cost, heuristic
        )
        trace.append(row)

    return {
        "path": [],
        "cost": float("inf"),
        "expanded": len(expanded),
        "expansion_order": expanded,
        "trace": trace,
    }

### c. Uji rekonstruksi dahulu

Ini contoh tiga state terpisah dari graph tugas. Jika belum diisi, fungsi masih
mengembalikan None; jika sudah diisi, assertion harus lolos.

In [4]:
example_parent = {"R": None, "T": "R", "U": "T"}
example_path = reconstruct_path(example_parent, "U")
if example_path is None:
    print("Lengkapi TODO R1-R2 terlebih dahulu.")
else:
    assert example_path == ["R", "T", "U"]
    print("Uji rekonstruksi lulus.")

Uji rekonstruksi lulus.


### d. Validator path — sudah disediakan

Fungsi ini memeriksa titik awal, titik akhir, edge yang dilewati, dan jumlah cost.
Anda tidak perlu mengubahnya. Validator tidak memberi jawaban path tugas.

In [5]:
def validate_path(problem, path, start, goal):
    assert path, "Path utama harus ditemukan."
    assert path[0] == start, "Path harus dimulai di start."
    assert path[-1] == goal, "Path harus berakhir di goal."
    total_cost = 0
    for current_state, next_state in zip(path, path[1:]):
        matching_cost = None
        for neighbor, step_cost in problem[current_state]:
            if neighbor == next_state:
                matching_cost = step_cost
                break
        assert matching_cost is not None, f"Edge ilegal: {current_state} -> {next_state}"
        total_cost += matching_cost
    return total_cost

### e. Validator hasil — sudah disediakan

Fungsi ini memeriksa kelengkapan field dan kecocokan trace dengan jumlah ekspansi.
Lulus validator belum membuktikan algoritma benar atau hasilnya optimal.
Kode dan analisis Anda tetap diperiksa.

In [6]:
def validate_result(problem, result, start, goal, algorithm, estimates):
    required_keys = {"path", "cost", "expanded", "expansion_order", "trace"}
    assert isinstance(result, dict), "Return harus berupa dictionary."
    assert required_keys <= result.keys(), "Field hasil belum lengkap."
    if result["path"]:
        assert result["cost"] == validate_path(problem, result["path"], start, goal)
    else:
        assert result["cost"] == float("inf"), "Failure harus memakai cost inf."
    assert isinstance(result["expanded"], int) and result["expanded"] >= 0
    assert result["expanded"] == len(result["expansion_order"])
    assert result["trace"], "Start harus tercatat pada trace."
    traced_expansions = []
    for iteration, row in enumerate(result["trace"], start=1):
        required_trace = {"iteration", "selected", "h", "action", "frontier"}
        if algorithm == "A*":
            required_trace.update({"g", "f", "updates"})
        assert required_trace <= row.keys(), "Field trace belum lengkap."
        assert row["iteration"] == iteration
        assert row["selected"] in problem
        assert row["h"] == estimates[row["selected"]]
        assert row["action"] in {"expand", "goal"}
        assert isinstance(row["frontier"], str)
        if row["action"] == "expand":
            traced_expansions.append(row["selected"])
        else:
            assert iteration == len(result["trace"]), "Goal hanya pada baris terakhir."
        if algorithm == "A*":
            assert row["f"] == row["g"] + row["h"]
            assert isinstance(row["updates"], list)
            for update in row["updates"]:
                fields = {"state", "previous_g", "new_g", "previous_parent", "new_parent"}
                assert fields <= update.keys(), "Catatan relaksasi belum lengkap."
                assert update["new_g"] < update["previous_g"]
                assert update["new_parent"] == row["selected"]
    assert traced_expansions == result["expansion_order"]
    if result["path"]:
        assert result["trace"][-1]["selected"] == goal
        assert result["trace"][-1]["action"] == "goal"
        if algorithm == "A*":
            assert result["trace"][-1]["g"] == result["cost"]
    else:
        assert all(row["action"] != "goal" for row in result["trace"])
    if algorithm == "Greedy":
        assert len(set(result["expansion_order"])) == result["expanded"]
    print("Validasi struktur, path, cost, dan field trace lulus.")

### f. Menampilkan hasil — sudah disediakan

Output wajib mencakup start, goal, algoritma, path, cost, jumlah ekspansi,
urutan ekspansi, dan trace. List dictionary diperbolehkan; DataFrame tidak wajib.

In [7]:
def show_result(result, algorithm, start, goal):
    print("Start State:", start)
    print("Goal State:", goal)
    print("Algorithm:", algorithm)
    print("Path:", " -> ".join(result["path"]))
    print("Total Path Cost:", result["cost"])
    print("Number of Expanded States (peristiwa):", result["expanded"])
    print("Expansion Order:", " -> ".join(result["expansion_order"]))
    print("\nTrace:")
    print_trace(result["trace"], algorithm)
    if algorithm == "A*":
        show_updates(result)

### g. Uji mini — disediakan

Uji ini memeriksa start=goal, unreachable/cycle, FIFO, dan duplicate.
Khusus A*, juga diperiksa perbaikan g, stale skip, dan reopen.
Expected cost hanya untuk graph mini, bukan graph 14 state.

Greedy mempertahankan path pertama; A* menggantinya jika ada path lebih murah.
Karena itu kedua algoritma memang memiliki expected cost berbeda pada uji duplicate.

In [8]:
def run_small_tests(search_function, algorithm):
    # Kasus kecil ini bukan graph tugas dan tidak mengungkap jawaban baseline.
    one_state = {"S": []}
    result = search_function(one_state, "S", "S", {"S": 0})
    assert result["path"] == ["S"] and result["cost"] == 0
    assert result["expanded"] == 0
    validate_result(one_state, result, "S", "S", algorithm, {"S": 0})

    cycle = {"S": [("P", 2)], "P": [("S", 2)], "G": []}
    cycle_h = {"S": 0, "P": 0, "G": 0}
    result = search_function(cycle, "S", "G", cycle_h)
    assert result["path"] == [] and result["cost"] == float("inf")
    assert result["expanded"] == 2
    validate_result(cycle, result, "S", "G", algorithm, cycle_h)

    tie_graph = {"S": [("Z", 1), ("A", 1)], "Z": [("G", 1)],
                 "A": [("G", 1)], "G": []}
    tie_h = {"S": 0, "Z": 0, "A": 0, "G": 0}
    result = search_function(tie_graph, "S", "G", tie_h)
    assert result["expansion_order"][1] == "Z", "Tie harus FIFO, bukan alfabetis."

    duplicate = {"S": [("P", 5), ("Q", 1)], "Q": [("P", 1)],
                 "P": [("G", 1)], "G": []}
    duplicate_h = {"S": 0, "P": 1, "Q": 0, "G": 0}
    result = search_function(duplicate, "S", "G", duplicate_h)
    validate_result(duplicate, result, "S", "G", algorithm, duplicate_h)
    if algorithm == "Greedy":
        assert result["cost"] == 6, "Greedy tugas mempertahankan path pertama."
    else:
        assert result["cost"] == 3, "A* harus memperbaiki g(P) dari 5 ke 2."
        assert any(update["state"] == "P" and update["previous_g"] == 5
                   and update["new_g"] == 2
                   for row in result["trace"] for update in row["updates"])

    if algorithm == "A*":
        stale = {"S": [("P", 5), ("Q", 1)], "Q": [("P", 1)],
                 "P": [("G", 10)], "G": []}
        stale_h = {"S": 0, "P": 0, "Q": 0, "G": 0}
        result = search_function(stale, "S", "G", stale_h)
        assert result["cost"] == 12
        assert result["expansion_order"].count("P") == 1
        assert len(result["trace"]) == result["expanded"] + 1
        validate_result(stale, result, "S", "G", algorithm, stale_h)

        reopen = {"S": [("P", 3), ("Q", 1)], "P": [("G", 4)],
                  "Q": [("P", 1)], "G": []}
        reopen_h = {"S": 0, "P": 0, "Q": 4, "G": 0}
        result = search_function(reopen, "S", "G", reopen_h)
        assert result["cost"] == 6
        assert result["expansion_order"].count("P") == 2
        validate_result(reopen, result, "S", "G", algorithm, reopen_h)
    print("Semua uji mini untuk", algorithm, "lulus.")

### h. Isi NRP → uji mini → jalankan baseline

Isi NRP sebagai string digit, kemudian jalankan cell ini. Pemilihan fungsi dilakukan
otomatis. Jika fungsi masih `pass`, Anda mendapat pesan TODO. Jika sudah diisi,
uji mini dijalankan **lebih dahulu**, kemudian baseline S → G.

Jika muncul `AssertionError`, baca pesan dan perbaiki tahap terkait; jangan mengubah
expected value atau validator agar tampak lulus. Jalankan kembali cell fungsi sebelum menguji.

In [9]:
nrp = "5025251054"
selected_search = None
algorithm = None
baseline_result = None

if not nrp or not nrp.isdigit():
    print("Isi nrp dengan digit NRP Anda, lalu jalankan ulang cell ini.")
else:
    if int(nrp[-1]) % 2 == 1:
        selected_search = greedy_search
        algorithm = "Greedy"
    else:
        selected_search = a_star_search
        algorithm = "A*"

    probe = selected_search({"S": []}, "S", "S", {"S": 0})
    if probe is None:
        print(f"{algorithm} belum diimplementasikan; lengkapi TODO terlebih dahulu.")
    else:
        run_small_tests(selected_search, algorithm)
        baseline_result = selected_search(graph, start, goal, heuristic)
        assert baseline_result["path"], "Graph baseline memiliki solusi."
        validate_result(graph, baseline_result, start, goal, algorithm, heuristic)
        show_result(baseline_result, algorithm, start, goal)

Validasi struktur, path, cost, dan field trace lulus.
Validasi struktur, path, cost, dan field trace lulus.
Validasi struktur, path, cost, dan field trace lulus.
Validasi struktur, path, cost, dan field trace lulus.
Validasi struktur, path, cost, dan field trace lulus.
Semua uji mini untuk A* lulus.
Validasi struktur, path, cost, dan field trace lulus.
Start State: S
Goal State: G
Algorithm: A*
Path: S -> C -> F -> J -> M -> G
Total Path Cost: 17
Number of Expanded States (peristiwa): 13
Expansion Order: S -> A -> C -> B -> F -> E -> D -> H -> J -> I -> K -> M -> L

Trace:


| Iterasi | Dipilih | g | h | f | Aksi | Frontier sesudah |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | S | 0 | 9 | 9 | expand | A(g=3,h=8,f=11); C(g=4,h=7,f=11); B(g=5,h=7,f=12) |
| 2 | A | 3 | 8 | 11 | expand | C(g=4,h=7,f=11); B(g=4,h=7,f=11); D(g=7,h=7,f=14); E(g=10,h=6,f=16) |
| 3 | C | 4 | 7 | 11 | expand | B(g=4,h=7,f=11); F(g=7,h=5,f=12); D(g=7,h=7,f=14); H(g=9,h=5,f=14); E(g=10,h=6,f=16) |
| 4 | B | 4 | 7 | 11 | expand | F(g=7,h=5,f=12); E(g=6,h=6,f=12); D(g=7,h=7,f=14); H(g=9,h=5,f=14) |
| 5 | F | 7 | 5 | 12 | expand | E(g=6,h=6,f=12); D(g=7,h=7,f=14); H(g=9,h=5,f=14); J(g=10,h=4,f=14); K(g=14,h=4,f=18) |
| 6 | E | 6 | 6 | 12 | expand | D(g=7,h=7,f=14); H(g=9,h=5,f=14); J(g=10,h=4,f=14); I(g=11,h=5,f=16); K(g=14,h=4,f=18) |
| 7 | D | 7 | 7 | 14 | expand | H(g=9,h=5,f=14); J(g=10,h=4,f=14); I(g=11,h=5,f=16); K(g=14,h=4,f=18) |
| 8 | H | 9 | 5 | 14 | expand | J(g=10,h=4,f=14); I(g=11,h=5,f=16); K(g=12,h=4,f=16) |
| 9 | J | 10 | 4 | 14 | expand | I(g=11,h=5,f=16); K(g=12,h=4,f=16); M(g=14,h=2,f=16); L(g=15,h=3,f=18) |
| 10 | I | 11 | 5 | 16 | expand | K(g=12,h=4,f=16); M(g=14,h=2,f=16); L(g=14,h=3,f=17) |
| 11 | K | 12 | 4 | 16 | expand | M(g=14,h=2,f=16); L(g=14,h=3,f=17) |
| 12 | M | 14 | 2 | 16 | expand | L(g=14,h=3,f=17); G(g=17,h=0,f=17) |
| 13 | L | 14 | 3 | 17 | expand | G(g=17,h=0,f=17) |
| 14 | G | 17 | 0 | 17 | goal | (kosong) |

| Iterasi | State | g lama | g baru | Parent lama | Parent baru |
| --- | --- | --- | --- | --- | --- |
| 2 | B | 5 | 4 | S | A |
| 4 | E | 10 | 6 | A | B |
| 8 | K | 14 | 12 | F | H |
| 10 | L | 15 | 14 | J | I |

### i. Jika program belum sesuai

| Gejala | Periksa |
|---|---|
| Loop tidak selesai pada cycle | Greedy: discovered/parent; A*: syarat `<`, bukan `<=` |
| Path benar tetapi cost berbeda | Cost berasal dari edge; biaya dan parent harus konsisten |
| Tie memilih A sebelum Z | Nomor urut harus mendahului state dalam tuple heap |
| Jumlah ekspansi terlalu besar | Jangan hitung goal atau stale pop; hitung reopen sebagai peristiwa |
| A* gagal uji reopen | Jangan memakai visited permanen untuk menolak g yang membaik |
| Semua baris frontier terlihat sama | Simpan snapshot string baru pada setiap iterasi |
| KeyError pada trace | Cocokkan nama field dengan kontrak bagian 3.d |

Lulus validator struktur belum membuktikan semua keputusan priority benar.
Periksa trace dan kode; setiap selected harus minimum priority menurut aturan algoritma.

## 5. Analisis Hasil

Jawab bagian sesuai algoritma Anda. Gunakan angka dan keputusan dari trace program.

### NRP ganjil — Greedy

1. Bagaimana state berikutnya dipilih dari frontier?
2. Apa peran heuristic? Mengapa g tidak menjadi priority?
3. Apakah path Anda minimum? Bandingkan cost dengan alternatif yang relevan.
4. Tunjukkan satu keputusan ketika heuristic memengaruhi pilihan. Sertakan minimal dua kandidat.
5. Apa kelemahan Greedy yang terlihat dari hasil Anda?

**Untuk nomor 3:** satu path lebih murah cukup untuk membantah optimalitas.
Satu path lebih mahal belum cukup untuk membuktikan hasil minimum; gunakan pembandingan
menyeluruh atau batas bawah yang sah jika Anda mengklaim minimum.

### NRP genap — A*

1. Bagaimana state berikutnya dipilih, termasuk ketika terjadi tie?
2. Jelaskan g, h, dan f menggunakan angka dari hasil Anda.
3. Tunjukkan satu perhitungan f=g+h dan bandingkan dengan kandidat pesaing.
4. Tunjukkan perbaikan biaya: g lama/baru, parent lama/baru, dan perlakuan entri usang.
5. Bagaimana heuristic memengaruhi kualitas hasil? Periksa consistency semua edge
   dan jelaskan hubungannya dengan h(G)=0 serta optimalitas.

Untuk kedua kelompok, jelaskan minimal dua pilihan node pada trace, termasuk tie jika ada.
Anda tidak diwajibkan mengimplementasikan algoritma kelompok lain.

Gunakan tabel `show_updates` untuk nomor 4. Catatan mencakup g lama/baru dan parent
lama/baru sehingga Anda tidak perlu menebak dari path akhir. Reopen tidak wajib terjadi
pada baseline yang consistent; hasil uji mini menunjukkan bahwa program mendukungnya.

### Jawaban saya

1. State berikutnya dipilih berdasarkan nilai f paling kecil, dengan rumus f = g + h. Kalau ada beberapa state dengan nilai f yang sama, program menggunakan urutan masuk ke frontier sebagai penentu.
Contohnya pada iterasi pertama, A dan C sama-sama punya f = 11, sedangkan B punya f = 12. Karena A masuk lebih dulu, A yang dipilih. Setelah itu C dan B juga sempat punya nilai f = 11, tetapi C dipilih lebih dulu karena masuk ke frontier lebih awal.
2. g adalah biaya yang sudah ditempuh dari state awal sampai state tersebut. h adalah perkiraan biaya dari state tersebut menuju goal. Sedangkan f adalah jumlah dari keduanya.
Contohnya pada state F:
g = 7 dan h = 5
Jadi:
f = g + h = 7 + 5 = 12
Nilai inilah yang digunakan A* untuk menentukan state yang akan dipilih berikutnya.
3. Pada iterasi ke-5, F dipilih dengan g = 7 dan h = 5, sehingga nilai f-nya adalah:
f(F) = 7 + 5 = 12
Pada saat yang sama E juga mempunyai f = 12, dengan g = 6 dan h = 6.
Jadi F dan E memiliki priority yang sama. F dipilih lebih dulu karena mengikuti aturan tie berdasarkan urutan masuk ke frontier.
4. Selama proses pencarian, ada beberapa state yang mendapat jalur baru dengan biaya yang lebih kecil. Contohnya B awalnya memiliki g = 5 dengan parent S. Setelah A diproses, ditemukan jalur S → A → B dengan g = 4, sehingga parent B berubah dari S menjadi A.
Perubahan lainnya yaitu E dari g = 10 menjadi g = 6, K dari 14 menjadi 12, dan L dari 15 menjadi 14.
Jadi A* tidak langsung mempertahankan jalur pertama yang ditemukan. Kalau ditemukan jalur yang lebih murah, nilai g dan parent akan diperbarui. Entri lama yang masih ada di frontier akan dianggap sudah tidak sesuai dengan nilai g terbaru dan dilewati.
5. Heuristic membantu A* memperkirakan state mana yang lebih dekat dengan goal. Dari hasil baseline, A* mendapatkan path S → C → F → J → M → G dengan total cost 17.
Heuristic yang digunakan juga memiliki h(G) = 0, sehingga pada goal nilai f sama dengan nilai g. Untuk consistency, pengecekan dilakukan pada setiap edge dengan membandingkan h state sekarang dengan biaya edge ditambah h state berikutnya. Kalau tidak ada pelanggaran consistency, heuristic tersebut consistent.
Dengan heuristic yang consistent, nilai f tidak akan menurun sepanjang jalur dan A* dapat mempertahankan sifat optimalnya selama aturan pengelolaan g dan frontier dijalankan dengan benar.

## 6. Eksperimen Heuristic

Semua mahasiswa melakukan satu eksperimen:

1. Salin heuristic awal dengan `.copy()`.
2. Ubah sekurangnya **tiga state non-goal**. Nilai tetap nonnegatif dan h(G)=0.
3. Jalankan algoritma yang sama.
4. Bandingkan expansion order, jumlah ekspansi, path, dan total cost.

Jangan mengubah graph, start, goal, urutan successor, atau aturan tie.
Jelaskan alasan hasil berubah menggunakan priority pada trace.
Jika hasil tidak berubah, jelaskan mengapa pilihan algoritma tetap sama.

Tuliskan perubahan pada `heuristic_changes = {"state": nilai_baru, ...}` di cell eksperimen.
Isi minimal tiga nama state non-goal yang ada pada graph. Program akan membuat salinan
heuristic, memeriksa nilainya, dan menjalankan dua hasil untuk dibandingkan.

**Catatan untuk analisis:** menurunkan heuristic admissible menjaga admissibility,
tetapi belum tentu menjaga consistency. Menaikkan nilainya belum tentu aman.
Jika belum bisa membuktikan admissibility, nyatakan bahwa statusnya belum terjamin.
Kegagalan consistency saja tidak membuktikan kegagalan admissibility.

| Aspek | Original | Modified |
|---|---|---|
| Nilai yang diubah | Tidak ada | A: 8 → 6, C: 7 → 5, F: 5 → 3 |
| Expansion order | S -> A -> C -> B -> F -> E -> D -> H -> J -> I -> K -> M -> L | S -> A -> C -> F -> B -> E -> D -> H -> J -> I -> K -> M -> L |
| Jumlah ekspansi | 13 | 13 |
| Path | S -> C -> F -> J -> M -> G | S -> C -> F -> J -> M -> G |
| Total cost | 17 | 17 |

In [10]:
def consistency_violations(problem, estimates):
    violations = []
    for current_state, successors in problem.items():
        for next_state, step_cost in successors:
            limit = step_cost + estimates[next_state]
            if estimates[current_state] > limit:
                violations.append((current_state, next_state,
                                   estimates[current_state], limit))
    return violations

In [11]:
heuristic_changes = {
    "A": 6,
    "C": 5,
    "F": 3
}
modified_heuristic = heuristic.copy()
for state, new_value in heuristic_changes.items():
    assert state in graph and state != goal, "Gunakan nama state non-goal yang valid."
    assert isinstance(new_value, (int, float)) and math.isfinite(new_value) and new_value >= 0
    modified_heuristic[state] = new_value

changed_states = []
for state in graph:
    if modified_heuristic[state] != heuristic[state]:
        changed_states.append(state)

modified_result = None
if selected_search is None or baseline_result is None:
    print("Selesaikan uji mini dan baseline terlebih dahulu.")
elif len(changed_states) < 3:
    print("Isi heuristic_changes agar sekurangnya tiga nilai benar-benar berubah.")
else:
    print("Pelanggaran consistency awal:", consistency_violations(graph, heuristic))
    print("Pelanggaran consistency modifikasi:",
          consistency_violations(graph, modified_heuristic))
    for state in changed_states:
        print(f"{state}: {heuristic[state]} -> {modified_heuristic[state]}")
    modified_result = selected_search(graph, start, goal, modified_heuristic)
    validate_result(graph, modified_result, start, goal, algorithm, modified_heuristic)
    print("\nOriginal heuristic:")
    show_result(baseline_result, algorithm, start, goal)
    print("\nModified heuristic:")
    show_result(modified_result, algorithm, start, goal)

Pelanggaran consistency awal: []
Pelanggaran consistency modifikasi: []
A: 8 -> 6
C: 7 -> 5
F: 5 -> 3
Validasi struktur, path, cost, dan field trace lulus.

Original heuristic:
Start State: S
Goal State: G
Algorithm: A*
Path: S -> C -> F -> J -> M -> G
Total Path Cost: 17
Number of Expanded States (peristiwa): 13
Expansion Order: S -> A -> C -> B -> F -> E -> D -> H -> J -> I -> K -> M -> L

Trace:


| Iterasi | Dipilih | g | h | f | Aksi | Frontier sesudah |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | S | 0 | 9 | 9 | expand | A(g=3,h=8,f=11); C(g=4,h=7,f=11); B(g=5,h=7,f=12) |
| 2 | A | 3 | 8 | 11 | expand | C(g=4,h=7,f=11); B(g=4,h=7,f=11); D(g=7,h=7,f=14); E(g=10,h=6,f=16) |
| 3 | C | 4 | 7 | 11 | expand | B(g=4,h=7,f=11); F(g=7,h=5,f=12); D(g=7,h=7,f=14); H(g=9,h=5,f=14); E(g=10,h=6,f=16) |
| 4 | B | 4 | 7 | 11 | expand | F(g=7,h=5,f=12); E(g=6,h=6,f=12); D(g=7,h=7,f=14); H(g=9,h=5,f=14) |
| 5 | F | 7 | 5 | 12 | expand | E(g=6,h=6,f=12); D(g=7,h=7,f=14); H(g=9,h=5,f=14); J(g=10,h=4,f=14); K(g=14,h=4,f=18) |
| 6 | E | 6 | 6 | 12 | expand | D(g=7,h=7,f=14); H(g=9,h=5,f=14); J(g=10,h=4,f=14); I(g=11,h=5,f=16); K(g=14,h=4,f=18) |
| 7 | D | 7 | 7 | 14 | expand | H(g=9,h=5,f=14); J(g=10,h=4,f=14); I(g=11,h=5,f=16); K(g=14,h=4,f=18) |
| 8 | H | 9 | 5 | 14 | expand | J(g=10,h=4,f=14); I(g=11,h=5,f=16); K(g=12,h=4,f=16) |
| 9 | J | 10 | 4 | 14 | expand | I(g=11,h=5,f=16); K(g=12,h=4,f=16); M(g=14,h=2,f=16); L(g=15,h=3,f=18) |
| 10 | I | 11 | 5 | 16 | expand | K(g=12,h=4,f=16); M(g=14,h=2,f=16); L(g=14,h=3,f=17) |
| 11 | K | 12 | 4 | 16 | expand | M(g=14,h=2,f=16); L(g=14,h=3,f=17) |
| 12 | M | 14 | 2 | 16 | expand | L(g=14,h=3,f=17); G(g=17,h=0,f=17) |
| 13 | L | 14 | 3 | 17 | expand | G(g=17,h=0,f=17) |
| 14 | G | 17 | 0 | 17 | goal | (kosong) |

| Iterasi | State | g lama | g baru | Parent lama | Parent baru |
| --- | --- | --- | --- | --- | --- |
| 2 | B | 5 | 4 | S | A |
| 4 | E | 10 | 6 | A | B |
| 8 | K | 14 | 12 | F | H |
| 10 | L | 15 | 14 | J | I |


Modified heuristic:
Start State: S
Goal State: G
Algorithm: A*
Path: S -> C -> F -> J -> M -> G
Total Path Cost: 17
Number of Expanded States (peristiwa): 13
Expansion Order: S -> A -> C -> F -> B -> E -> D -> H -> J -> I -> K -> M -> L

Trace:


| Iterasi | Dipilih | g | h | f | Aksi | Frontier sesudah |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | S | 0 | 9 | 9 | expand | A(g=3,h=6,f=9); C(g=4,h=5,f=9); B(g=5,h=7,f=12) |
| 2 | A | 3 | 6 | 9 | expand | C(g=4,h=5,f=9); B(g=4,h=7,f=11); D(g=7,h=7,f=14); E(g=10,h=6,f=16) |
| 3 | C | 4 | 5 | 9 | expand | F(g=7,h=3,f=10); B(g=4,h=7,f=11); D(g=7,h=7,f=14); H(g=9,h=5,f=14); E(g=10,h=6,f=16) |
| 4 | F | 7 | 3 | 10 | expand | B(g=4,h=7,f=11); D(g=7,h=7,f=14); H(g=9,h=5,f=14); J(g=10,h=4,f=14); E(g=9,h=6,f=15); K(g=14,h=4,f=18) |
| 5 | B | 4 | 7 | 11 | expand | E(g=6,h=6,f=12); D(g=7,h=7,f=14); H(g=9,h=5,f=14); J(g=10,h=4,f=14); K(g=14,h=4,f=18) |
| 6 | E | 6 | 6 | 12 | expand | D(g=7,h=7,f=14); H(g=9,h=5,f=14); J(g=10,h=4,f=14); I(g=11,h=5,f=16); K(g=14,h=4,f=18) |
| 7 | D | 7 | 7 | 14 | expand | H(g=9,h=5,f=14); J(g=10,h=4,f=14); I(g=11,h=5,f=16); K(g=14,h=4,f=18) |
| 8 | H | 9 | 5 | 14 | expand | J(g=10,h=4,f=14); I(g=11,h=5,f=16); K(g=12,h=4,f=16) |
| 9 | J | 10 | 4 | 14 | expand | I(g=11,h=5,f=16); K(g=12,h=4,f=16); M(g=14,h=2,f=16); L(g=15,h=3,f=18) |
| 10 | I | 11 | 5 | 16 | expand | K(g=12,h=4,f=16); M(g=14,h=2,f=16); L(g=14,h=3,f=17) |
| 11 | K | 12 | 4 | 16 | expand | M(g=14,h=2,f=16); L(g=14,h=3,f=17) |
| 12 | M | 14 | 2 | 16 | expand | L(g=14,h=3,f=17); G(g=17,h=0,f=17) |
| 13 | L | 14 | 3 | 17 | expand | G(g=17,h=0,f=17) |
| 14 | G | 17 | 0 | 17 | goal | (kosong) |

| Iterasi | State | g lama | g baru | Parent lama | Parent baru |
| --- | --- | --- | --- | --- | --- |
| 2 | B | 5 | 4 | S | A |
| 4 | E | 10 | 9 | A | F |
| 5 | E | 9 | 6 | F | B |
| 8 | K | 14 | 12 | F | H |
| 10 | L | 15 | 14 | J | I |

### Analisis eksperimen saya

- Alasan memilih tiga (atau lebih) perubahan nilai: Saya menurunkan nilai heuristic pada state A, C, dan F untuk melihat pengaruh perubahan heuristic terhadap prioritas pemilihan state pada A*. Nilainya diubah dari A: 8 menjadi 6, C: 7 menjadi 5, dan F: 5 menjadi 3.
- Perubahan priority pada iterasi yang relevan: Pada heuristic awal, setelah C diekspansi, B memiliki f=11 sedangkan F memiliki f=12, sehingga B dipilih lebih dulu. Setelah heuristic F diturunkan menjadi 3, nilai F menjadi f=7+3=10, sehingga F dipilih lebih dulu daripada B yang memiliki f=11. Hal ini membuat urutan ekspansi F dan B bertukar.
- Perbandingan expansion order dan jumlah ekspansi: Expansion order berubah dari S -> A -> C -> B -> F -> E -> D -> H -> J -> I -> K -> M -> L menjadi S -> A -> C -> F -> B -> E -> D -> H -> J -> I -> K -> M -> L. Walaupun urutannya berubah, jumlah ekspansi tetap sama, yaitu 13 state.
- Perbandingan path dan total cost: Path yang dihasilkan pada kedua percobaan tetap sama, yaitu S -> C -> F -> J -> M -> G, dengan total cost 17. Jadi perubahan heuristic pada eksperimen ini tidak mengubah solusi akhir maupun biaya solusi.
- Pemeriksaan admissibility/consistency dan batas kesimpulan: Hasil pemeriksaan menunjukkan Pelanggaran consistency awal: [] dan Pelanggaran consistency modifikasi: [], sehingga tidak ditemukan pelanggaran consistency pada kedua heuristic. Namun, hasil tersebut belum cukup untuk membuktikan admissibility secara langsung. Jadi kesimpulan yang dapat diambil adalah heuristic modifikasi tetap konsisten pada graph ini, tetapi status admissibility tidak dinyatakan hanya berdasarkan hasil pemeriksaan tersebut.

## 7. Ketentuan dan Pengumpulan

- Kerjakan secara individual; implementasikan sendiri satu algoritma sesuai NRP.
- `heapq` diperbolehkan. Dilarang memakai solver Greedy/A*, shortest-path library,
  atau mengimpor/menyalin fungsi search lengkap dari materi sebagai jawaban.
  Materi boleh dipakai untuk memahami pola; helper yang disediakan boleh dipakai langsung.
- `matplotlib` dan `networkx` hanya untuk visualisasi opsional.
- Jangan hard-code path, cost, atau urutan ekspansi.
- Gunakan nama variabel jelas dan komentar secukupnya. OOP kompleks tidak diperlukan.
- Kumpulkan `NRP_Nama_InformedSearch.ipynb` dengan identitas, kode, output baseline,
  trace, analisis, eksperimen, dan hasil uji tambahan.
- Sebelum dikumpulkan, lakukan **Restart Kernel and Run All Cells**, lalu simpan output.

Tidak ada expected solution path atau cost akhir graph tugas yang disediakan.

**Sebelum mengumpulkan:** isi identitas dan NRP; R1–R2 serta TODO algoritma terpilih
selesai; uji mini lulus; baseline dan trace tersimpan; lima jawaban analisis terisi;
minimal tiga perubahan heuristic benar-benar diterapkan; tabel/uraian perbandingan terisi.
Fungsi algoritma kelompok lain boleh tetap `pass`.

## 8. Rubrik Penilaian

| Kategori | Poin | Yang dinilai |
|---|---:|---|
| Correctness algoritma | 35 | Priority dan tie (12), successor dan biaya (8), goal/failure/start=goal (7), rekonstruksi path (8) |
| Frontier dan state management | 20 | Priority queue (5), duplicate/cycle handling (10), parent tracking (5) |
| Output dan tracing | 15 | Field hasil lengkap (5), trace dan hitungan ekspansi benar (10) |
| Analisis dan eksperimen | 20 | Analisis algoritma dengan bukti (10), perbandingan heuristic dan penjelasan (10) |
| Code quality | 10 | Keterbacaan (5), Run All dan uji tambahan (5) |
| **Total** | **100** | |

**Penilaian sesuai algoritma:** pada duplicate handling, Greedy dinilai atas pencegahan
duplicate/cycle dan pemeliharaan path pertama. A* dinilai atas perbaikan g, stale skip,
dan reopen. Greedy tidak diwajibkan melakukan pembaruan g seperti A*.
Tidak ada bonus untuk kode lebih rumit atau mengerjakan kedua algoritma.